In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes per rules
df_metric["timestamp"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
df_trace["timestamp"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)
df_log["timestamp"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)
df_err["timestamp"] = pd.to_datetime(df_err["timestamp"], unit="s", utc=True)

# Aggregate metric.csv by (cmdb_id, kpi_name): count and global P95 of 'value'
metric_agg = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(
        count=("value", "size"),
        p95=("value", lambda x: float(x.quantile(0.95)) if not x.empty else float("nan"))
    )
)
metric_agg["p95"] = metric_agg["p95"].round(6)
metric_agg_out = metric_agg.sort_values(["cmdb_id", "kpi_name"]).head(50)[["cmdb_id", "kpi_name", "count", "p95"]]

# Aggregate trace.csv by (cmdb_id, trace_name): count and global P95 of 'value'
trace_agg = (
    df_trace
    .groupby(["cmdb_id", "trace_name"], as_index=False)
    .agg(
        count=("value", "size"),
        p95=("value", lambda x: float(x.quantile(0.95)) if not x.empty else float("nan"))
    )
)
trace_agg["p95"] = trace_agg["p95"].round(6)
trace_agg_out = trace_agg.sort_values(["cmdb_id", "trace_name"]).head(50)[["cmdb_id", "trace_name", "count", "p95"]]

# Aggregate log.csv by (cmdb_id, log_name): count and global P95 of 'value'
log_agg = (
    df_log
    .groupby(["cmdb_id", "log_name"], as_index=False)
    .agg(
        count=("value", "size"),
        p95=("value", lambda x: float(x.quantile(0.95)) if not x.empty else float("nan"))
    )
)
log_agg["p95"] = log_agg["p95"].round(6)
log_agg_out = log_agg.sort_values(["cmdb_id", "log_name"]).head(50)[["cmdb_id", "log_name", "count", "p95"]]

# Summarize error_logs.csv: distinct cmdb_id with counts and up to 5 sample messages
# join up to first 5 messages with separator; keep reasonably compact
err_samples = (
    df_err
    .groupby("cmdb_id", as_index=False)
    .agg(
        count_messages=("message", "size"),
        sample_messages=("message", lambda s: " || ".join(s.astype(str).head(5)))
    )
)
# Truncate sample_messages to keep output compact (e.g., 1000 chars)
err_samples["sample_messages"] = err_samples["sample_messages"].str.slice(0, 1000)
error_summary_out = err_samples.sort_values("cmdb_id").head(50)[["cmdb_id", "count_messages", "sample_messages"]]

# Display the four compact outputs (IPython will show them)
metric_agg_out, trace_agg_out, log_agg_out, error_summary_out

```
Out[1]:
```
Summary of the aggregated telemetry (straightforward highlights):

- What I loaded and aggregated:
  - metric.csv, trace.csv, log.csv, error_logs.csv
  - For metric/trace/log: grouped by (cmdb_id, kpi/trace/log name) and computed count and global P95 of value (rounded to 6 decimals).
  - For error_logs: distinct cmdb_id, message counts, and up to 5 sample messages.

- Metric highlights (first 50 rows shown; most groups have 25 samples):
  - adservice.diskio — P95 ≈ 112,750.500000 (very large)
  - paymentservice.mem — P95 ≈ 39,834,110.000000 (very large)
  - paymentservice.socket — P95 = 3.000000
  - CPU P95 values generally < 1 for many services (e.g., adservice.cpu ≈ 0.939048)
  - Most metric groups have count = 25 (some components may have different counts like redis with 4 in logs).

- Trace highlights (first 50 rows shown):
  - checkoutservice.trace.from_frontendservice.duration_mean — P95 ≈ 0.714538 (notable latency)
  - Some trace row_count P95 values are large (e.g., checkoutservice row_count ≈ 273.8)
  - Many trace error_rate P95 values are 0.0 for multiple service-pairs.

- Log highlights (first 50 rows shown):
  - frontend.log.error_count — P95 ≈ 2.800000 (non-zero errors)
  - frontend.log.row_count — P95 ≈ 2705.600000 (high request/row volume)
  - Many services have log.error_count P95 = 0.0 (no frequent logged errors in those groups).

- Error logs summary:
  - frontend — 16 messages (sample messages show repeated "failed to retrieve ads" entries).
  - (Other cmdb_id rows may exist but the visible summary shows frontend as the primary source of collected error messages.)

- Quick interpretation (straightforward):
  - Frontend has concentrated error logs (16 messages) mentioning "failed to retrieve ads" and non-zero log.error_count (P95 ≈ 2.8) — investigate frontend → adservice interaction.
  - adservice shows extremely high disk I/O P95 — investigate disk I/O on adservice.
  - paymentservice shows very large memory P95 and a socket value — investigate paymentservice memory usage and socket behavior.
  - Several traces show elevated latency on checkoutservice when called from frontend — worth investigating frontend→checkoutservice latency path.

If you want, I can (next):
- show specific top metric/trace/log rows sorted by P95,
- filter these aggregates to a particular incident time window,
- or list the full distinct cmdb_id values present in error_logs.

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id    kpi_name  count           p95
0        adservice         cpu     25  9.390480e-01
1        adservice      diskio     25  1.127505e+05
2        adservice       error     25  4.133000e-02
3        adservice  latency-50     25  3.141000e-03
4        adservice  latency-90     25  4.898000e-03
..             ...         ...    ...           ...
45  paymentservice         cpu     25  7.643720e-01
46  paymentservice  latency-50     25  3.458000e-03
47  paymentservice  latency-90     25  6.982000e-03
48  paymentservice         mem     25  3.983411e+07
49  paymentservice      socket     25  3.000000e+00

[50 rows x 4 columns],             cmdb_id                                trace_name  count         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.015002
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    0.007212
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25  273.800000
4   checkoutservice  trace.from_frontendservice.duration_mean     25    0.714538
..              ...                                       ...    ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95     25    0.000735
46     emailservice     trace.from_checkoutservice.error_rate     25    0.000000
47     emailservice      trace.from_checkoutservice.row_count     25   33.800000
48     emailservice             trace.from_root.duration_mean     25    0.000048
49     emailservice              trace.from_root.duration_p95     25    0.000081

[50 rows x 4 columns],                   cmdb_id         log_name  count     p95
0               adservice  log.error_count     25     0.0
1               adservice    log.row_count     25   455.6
2             cartservice  log.error_count     25     0.0
3             cartservice    log.row_count     25   789.0
4         checkoutservice  log.error_count     25     0.0
5         checkoutservice    log.row_count     25   100.6
6         currencyservice  log.error_count     25     0.0
7         currencyservice    log.row_count     25  2434.0
8            emailservice  log.error_count     25     0.0
9            emailservice    log.row_count     25    33.8
10               frontend  log.error_count     25     2.8
11               frontend    log.row_count     25  2705.6
12         paymentservice  log.error_count     25     0.0
13         paymentservice    log.row_count     25    67.6
14  recommendationservice  log.error_count     25     0.0
15  recommendationservice    log.row_count     25   585.4
16                  redis  log.error_count      4     0.0
17                  redis    log.row_count      4     5.0
18        shippingservice  log.error_count     25     0.0
19        shippingservice    log.row_count     25   477.6,     cmdb_id  count_messages                                    sample_messages
0  frontend              16  failed to retrieve ads || failed to retrieve a...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse df_metric if it exists from previous steps; otherwise load it
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp is parsed as UTC datetime per rules
if not pd.api.types.is_datetime64_any_dtype(df_metric["timestamp"]):
    df_metric["timestamp"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# 1) Calculate global P95 (using entire series) for each (cmdb_id, kpi_name)
metric_p95 = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(
        count_total=("value", "size"),
        global_p95=("value", lambda s: float(s.quantile(0.95)) if not s.empty else float("nan"))
    )
)
metric_p95["global_p95"] = metric_p95["global_p95"].round(6)

# 2) Parse timestamps (already done) and filter to the incident window (UTC)
start = pd.to_datetime("2024-01-20 11:09:49", utc=True)
end = pd.to_datetime("2024-01-20 11:39:49", utc=True)
df_window = df_metric[(df_metric["timestamp"] >= start) & (df_metric["timestamp"] <= end)].copy()

# 3) Join the filtered rows with the global-P95 table
dfw = df_window.merge(metric_p95[["cmdb_id", "kpi_name", "global_p95"]], on=["cmdb_id", "kpi_name"], how="left")

# 4) Identify anomaly rows:
#    value >= global_p95 OR (global_p95 == 0 and value > 0)
# Handle potential NaNs conservatively (treat NaN global_p95 as non-anomalous)
p95 = dfw["global_p95"]
anomaly_mask = (~p95.isna()) & ((dfw["value"] >= p95) | ((p95 == 0) & (dfw["value"] > 0)))
anomalies = dfw[anomaly_mask].copy()

# 5) Aggregate anomaly rows by (cmdb_id, kpi_name)
if anomalies.empty:
    anomaly_agg = pd.DataFrame(columns=[
        "cmdb_id", "kpi_name", "anomaly_count",
        "earliest_anomaly_timestamp", "max_value_in_window",
        "global_p95", "severity"
    ])
else:
    agg = (
        anomalies
        .groupby(["cmdb_id", "kpi_name"], as_index=False)
        .agg(
            anomaly_count=("value", "size"),
            earliest_anomaly_timestamp=("timestamp", "min"),
            max_value_in_window=("value", "max"),
            global_p95=("global_p95", "first")
        )
    )
    # severity: if global_p95 == 0 -> severity = max_value_in_window, else (max - p95)/p95
    def compute_sev(row):
        gp = row["global_p95"]
        mv = row["max_value_in_window"]
        if gp == 0:
            return float(mv)
        else:
            # protect against division by zero already handled; gp != 0 here
            return float((mv - gp) / gp)
    agg["severity"] = agg.apply(compute_sev, axis=1)
    # Format earliest timestamp as ISO UTC string for compactness
    agg["earliest_anomaly_timestamp"] = agg["earliest_anomaly_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    # Round numeric outputs to 6 decimals
    agg["max_value_in_window"] = agg["max_value_in_window"].round(6)
    agg["global_p95"] = agg["global_p95"].round(6)
    agg["severity"] = agg["severity"].round(6)
    # Sort by severity descending and limit to top 20
    anomaly_agg = agg.sort_values("severity", ascending=False).head(20).reset_index(drop=True)

# 6) Count distinct anomalous (cmdb_id, kpi_name) pairs
distinct_anomaly_pairs = anomaly_agg.shape[0]

# 7) Return the aggregated anomaly table (top 20) and the distinct anomaly-pair count
anomaly_agg, distinct_anomaly_pairs

```
Out[2]:
```
```python
# Build a concise plain-English summary from the previously computed anomaly_agg and distinct_anomaly_pairs
try:
    anomaly_agg  # rely on existing variable from prior execution
    distinct_anomaly_pairs
except NameError:
    summary = "Anomaly results are not available in the current session."
else:
    top_n = min(5, len(anomaly_agg))
    lines = []
    lines.append(f"Distinct anomalous (cmdb_id, kpi_name) pairs: {int(distinct_anomaly_pairs)}")
    lines.append(f"Top {top_n} anomalies by severity:")
    for i, row in enumerate(anomaly_agg.head(top_n).itertuples(index=False), start=1):
        cmdb = row.cmdb_id
        kpi = row.kpi_name
        count = int(row.anomaly_count)
        ts = row.earliest_anomaly_timestamp
        maxv = float(row.max_value_in_window)
        p95 = float(row.global_p95)
        sev = float(row.severity)
        lines.append(
            f"{i}) {cmdb} | {kpi} — severity={sev:.6f}, max_value={maxv:.6f}, global_p95={p95:.6f}, "
            f"anomaly_count={count}, earliest={ts}"
        )
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window     global_p95       severity
0         currencyservice      diskio             25       2024-01-20T11:12:00Z        232900.645404       0.000000  232900.645404
1                   redis      diskio              2       2024-01-20T11:12:00Z         17961.234310    9736.268050       0.844776
2               adservice       error              2       2024-01-20T11:28:00Z             0.066750       0.041330       0.615050
3               adservice      diskio              2       2024-01-20T11:27:00Z        166445.047846  112750.522549       0.476224
4         checkoutservice  latency-50              2       2024-01-20T11:31:00Z             0.423380       0.374330       0.131033
5         checkoutservice    workload              2       2024-01-20T11:17:00Z             1.190067       1.084893       0.096944
6            emailservice    workload              2       2024-01-20T11:17:00Z             1.178867       1.096167       0.075444
7         currencyservice  latency-90              2       2024-01-20T11:29:00Z             0.481030       0.448197       0.073257
8         currencyservice      socket             11       2024-01-20T11:26:00Z             6.433333       6.000000       0.072222
9             cartservice         cpu              2       2024-01-20T11:12:00Z             3.061585       2.860008       0.070481
10         paymentservice    workload              2       2024-01-20T11:17:00Z             1.166817       1.095707       0.064898
11               frontend  latency-50              2       2024-01-20T11:31:00Z             0.391562       0.368134       0.063640
12               frontend  latency-90              2       2024-01-20T11:31:00Z             1.805520       1.697960       0.063346
13              adservice         cpu              2       2024-01-20T11:15:00Z             0.983076       0.939048       0.046886
14  recommendationservice         cpu              2       2024-01-20T11:12:00Z             3.055323       2.920091       0.046311
15        shippingservice    workload              2       2024-01-20T11:17:00Z             8.208933       7.860963       0.044266
16        currencyservice  latency-50              2       2024-01-20T11:31:00Z             0.078249       0.074936       0.044211
17  productcatalogservice         cpu              2       2024-01-20T11:12:00Z             4.281936       4.109580       0.041940
18         paymentservice  latency-90              2       2024-01-20T11:21:00Z             0.007261       0.006982       0.039945
19  recommendationservice  latency-90              2       2024-01-20T11:31:00Z             0.022897       0.022176       0.032510, 20)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_trace if available, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed as UTC datetime (timestamps are Unix seconds)
if not pd.api.types.is_datetime64_any_dtype(df_trace["timestamp"]):
    df_trace["timestamp"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)

# 1) Compute global P95 for each (cmdb_id, trace_name) using the full series
trace_p95 = (
    df_trace
    .groupby(["cmdb_id", "trace_name"], as_index=False)
    .agg(
        count_total=("value", "size"),
        global_p95=("value", lambda s: float(s.quantile(0.95)) if not s.empty else float("nan"))
    )
)
trace_p95["global_p95"] = trace_p95["global_p95"].round(6)

# 2) Filter trace rows to the incident window (UTC)
start = pd.to_datetime("2024-01-20 11:09:49", utc=True)
end = pd.to_datetime("2024-01-20 11:39:49", utc=True)
df_trace_window = df_trace[(df_trace["timestamp"] >= start) & (df_trace["timestamp"] <= end)].copy()

# 3) Join filtered rows with global-P95 table
dfw = df_trace_window.merge(trace_p95[["cmdb_id", "trace_name", "global_p95"]], on=["cmdb_id", "trace_name"], how="left")

# 4) Identify anomaly rows:
# value >= global_p95 OR (global_p95 == 0 and value > 0). Ignore NaN global_p95 (treat as non-anomalous)
p95 = dfw["global_p95"]
anomaly_mask = (~p95.isna()) & ((dfw["value"] >= p95) | ((p95 == 0) & (dfw["value"] > 0)))
anomalies = dfw[anomaly_mask].copy()

# 5) Aggregate anomalies by (cmdb_id, trace_name)
if anomalies.empty:
    anomaly_agg = pd.DataFrame(columns=[
        "cmdb_id", "trace_name", "anomaly_count",
        "earliest_anomaly_timestamp", "max_value_in_window",
        "global_p95", "severity"
    ])
else:
    agg = (
        anomalies
        .groupby(["cmdb_id", "trace_name"], as_index=False)
        .agg(
            anomaly_count=("value", "size"),
            earliest_anomaly_timestamp=("timestamp", "min"),
            max_value_in_window=("value", "max"),
            global_p95=("global_p95", "first")
        )
    )
    def compute_sev(row):
        gp = row["global_p95"]
        mv = row["max_value_in_window"]
        if gp == 0:
            return float(mv)
        else:
            return float((mv - gp) / gp)
    agg["severity"] = agg.apply(compute_sev, axis=1)
    # Format timestamp and round numeric outputs
    agg["earliest_anomaly_timestamp"] = agg["earliest_anomaly_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    agg["max_value_in_window"] = agg["max_value_in_window"].round(6)
    agg["global_p95"] = agg["global_p95"].round(6)
    agg["severity"] = agg["severity"].round(6)
    anomaly_agg = agg.sort_values("severity", ascending=False).head(20).reset_index(drop=True)

# 6) Count distinct anomalous pairs
distinct_anomaly_pairs = int(anomaly_agg.shape[0])

# 7) Return compact results (dataframe limited to top 20 and the distinct count)
anomaly_agg, distinct_anomaly_pairs

```
Out[3]:
```
```python
# Build a concise plain-English summary from the previously computed trace anomaly_agg and distinct_anomaly_pairs
try:
    anomaly_agg  # rely on existing variable from prior execution
    distinct_anomaly_pairs
except NameError:
    summary = "Trace anomaly results are not available in the current session."
else:
    top_n = min(5, len(anomaly_agg))
    lines = []
    lines.append(f"Distinct anomalous (cmdb_id, trace_name) pairs: {int(distinct_anomaly_pairs)}.")
    lines.append(f"Top {top_n} trace anomalies by severity (severity, max_value, global_p95, earliest_timestamp):")
    for i, row in enumerate(anomaly_agg.head(top_n).itertuples(index=False), start=1):
        cmdb = row.cmdb_id
        trace = row.trace_name
        sev = float(row.severity)
        maxv = float(row.max_value_in_window)
        p95 = float(row.global_p95)
        ts = row.earliest_anomaly_timestamp
        lines.append(
            f"{i}) {cmdb} | {trace} — severity={sev:.6f}, max_value={maxv:.6f}, global_p95={p95:.6f}, earliest={ts}"
        )
    lines.append("Interpretation: very high severities at the top indicate trace duration spikes affecting recommendationservice and its calls (and related root→recommendationservice traces). Several checkoutservice↔currencyservice traces also show elevated duration percentiles, indicating latency on checkout→currency paths.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                    trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95   severity
0   recommendationservice                 trace.from_root.duration_mean              2       2024-01-20T11:17:00Z             0.003562    0.000071  49.163732
1                    root  trace.to_recommendationservice.duration_mean              2       2024-01-20T11:17:00Z             0.003562    0.000071  49.163732
2         checkoutservice         trace.to_currencyservice.duration_p95              2       2024-01-20T11:12:00Z             0.018750    0.000971  18.309990
3         currencyservice       trace.from_checkoutservice.duration_p95              2       2024-01-20T11:12:00Z             0.018750    0.000971  18.309990
4         checkoutservice         trace.to_checkoutservice.duration_p95              2       2024-01-20T11:28:00Z             0.025790    0.007212   2.575950
5         checkoutservice       trace.from_checkoutservice.duration_p95              2       2024-01-20T11:28:00Z             0.025790    0.007212   2.575950
6         currencyservice      trace.from_checkoutservice.duration_mean              2       2024-01-20T11:12:00Z             0.006600    0.003279   1.012809
7         checkoutservice        trace.to_currencyservice.duration_mean              2       2024-01-20T11:12:00Z             0.006600    0.003279   1.012809
8                    root   trace.to_recommendationservice.duration_p95              2       2024-01-20T11:17:00Z             0.000298    0.000156   0.911859
9   recommendationservice                  trace.from_root.duration_p95              2       2024-01-20T11:17:00Z             0.000298    0.000156   0.911859
10         paymentservice       trace.from_checkoutservice.duration_p95              2       2024-01-20T11:21:00Z             0.001015    0.000546   0.858791
11        checkoutservice          trace.to_paymentservice.duration_p95              2       2024-01-20T11:21:00Z             0.001015    0.000546   0.858791
12  productcatalogservice                  trace.from_root.duration_p95              2       2024-01-20T11:33:00Z             0.094709    0.056552   0.674729
13                   root   trace.to_productcatalogservice.duration_p95              2       2024-01-20T11:33:00Z             0.094709    0.056552   0.674729
14                   root          trace.to_paymentservice.duration_p95              2       2024-01-20T11:14:00Z             0.000655    0.000400   0.636750
15         paymentservice                  trace.from_root.duration_p95              2       2024-01-20T11:14:00Z             0.000655    0.000400   0.636750
16        currencyservice                  trace.from_root.duration_p95              2       2024-01-20T11:14:00Z             0.000215    0.000150   0.431667
17                   root         trace.to_currencyservice.duration_p95              2       2024-01-20T11:14:00Z             0.000215    0.000150   0.431667
18                   root         trace.to_paymentservice.duration_mean              2       2024-01-20T11:14:00Z             0.000165    0.000119   0.383754
19         paymentservice                 trace.from_root.duration_mean              2       2024-01-20T11:14:00Z             0.000165    0.000119   0.383754, 20)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse df_trace if available, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp is parsed as UTC datetime
if not pd.api.types.is_datetime64_any_dtype(df_trace["timestamp"]):
    df_trace["timestamp"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-20 11:09:49", utc=True)
end = pd.to_datetime("2024-01-20 11:39:49", utc=True)

# Filter to incident window
df_tw = df_trace[(df_trace["timestamp"] >= start) & (df_trace["timestamp"] <= end)].copy()

# Define selection patterns per request
# 1) checkoutservice where trace_name contains 'to_currencyservice' or 'from_checkoutservice'
cond_checkout = (
    (df_tw["cmdb_id"] == "checkoutservice") &
    df_tw["trace_name"].str.contains(r"to_currencyservice|from_checkoutservice", regex=True)
)

# 2) currencyservice where trace_name contains 'from_checkoutservice' or 'from_root' or 'from_checkoutservice.duration'
# (using 'from_checkoutservice' covers both with/without '.duration')
cond_currency = (
    (df_tw["cmdb_id"] == "currencyservice") &
    df_tw["trace_name"].str.contains(r"from_checkoutservice|from_root", regex=True)
)

# Combined filter
df_sel = df_tw[cond_checkout | cond_currency].copy()

# Sort by timestamp and limit to top 50 rows
df_sel = df_sel.sort_values("timestamp").reset_index(drop=True)
filtered_rows_out = df_sel.loc[:, ["timestamp", "cmdb_id", "trace_name", "value"]].head(50).copy()
# Format timestamp as ISO UTC string for compact display
filtered_rows_out["timestamp"] = filtered_rows_out["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
filtered_rows_out["value"] = filtered_rows_out["value"].round(6)

# Aggregate summary per (cmdb_id, trace_name) for the returned rows
if df_sel.empty:
    agg_summary_out = pd.DataFrame(columns=[
        "cmdb_id", "trace_name", "count",
        "earliest_timestamp", "latest_timestamp",
        "min_value", "max_value", "mean_value"
    ])
else:
    agg = (
        df_sel
        .groupby(["cmdb_id", "trace_name"], as_index=False)
        .agg(
            count=("value", "size"),
            earliest_timestamp=("timestamp", "min"),
            latest_timestamp=("timestamp", "max"),
            min_value=("value", "min"),
            max_value=("value", "max"),
            mean_value=("value", "mean")
        )
    )
    # Format timestamps and numeric precision
    agg["earliest_timestamp"] = agg["earliest_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    agg["latest_timestamp"] = agg["latest_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    agg["min_value"] = agg["min_value"].round(6)
    agg["max_value"] = agg["max_value"].round(6)
    agg["mean_value"] = agg["mean_value"].round(6)
    # Keep compact and limited to at most 50 rows
    agg_summary_out = agg.sort_values(["cmdb_id", "trace_name"]).head(50).reset_index(drop=True)

# Return the filtered rows and the aggregate summary (both DataFrames)
filtered_rows_out, agg_summary_out

```
Out[4]:
```
```python
# Build a plain-English summary string from the previously computed filtered_rows_out and agg_summary_out
try:
    filtered_rows_out
    agg_summary_out
except NameError:
    summary = "Requested filtered trace results are not available in the current session."
else:
    lines = []
    lines.append("Trace rows filtered to 2024-01-20 11:09:49 - 11:39:49 UTC (selected checkoutservice & currencyservice patterns):")
    lines.append(f"- Returned trace rows (limited view): {int(filtered_rows_out.shape[0])} rows (top 50 shown).")
    lines.append(f"- Distinct (cmdb_id, trace_name) groups in summary: {int(agg_summary_out.shape[0])}.")
    lines.append("")
    lines.append("Aggregate summary per (cmdb_id, trace_name):")
    for r in agg_summary_out.itertuples(index=False):
        cmdb = r.cmdb_id
        tname = r.trace_name
        cnt = int(r.count)
        et = r.earliest_timestamp
        lt = r.latest_timestamp
        mn = f"{r.min_value:.6f}"
        mx = f"{r.max_value:.6f}"
        mean = f"{r.mean_value:.6f}"
        lines.append(f"- {cmdb} | {tname} — count={cnt}, earliest={et}, latest={lt}, min={mn}, max={mx}, mean={mean}")
    lines.append("")
    lines.append("Quick interpretation:")
    lines.append("- checkoutservice shows multiple 'from_checkoutservice' traces with duration_mean values around 0.0036–0.017921s (mean ~0.005453) and a row_count up to 290.")
    lines.append("- checkoutservice → currencyservice traces have small duration means (up to ~0.006600s) but non-trivial row_count (up to 120, mean ~87.96).")
    lines.append("- currencyservice 'from_checkoutservice' traces mirror similar duration ranges and row_counts (min 11, max 120, mean ~87.96).")
    lines.append("- currencyservice 'from_root' durations are very small (max ~0.000215s, mean ~0.000054s) with row_count 4–14 (mean ~11.52).")
    lines.append("")
    lines.append("If you want the actual filtered rows (up to 50) or to further inspect any specific (cmdb_id, trace_name), tell me which one and I will show the rows or more detailed stats.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               timestamp          cmdb_id                                trace_name      value
0   2024-01-20T11:12:00Z  checkoutservice  trace.from_checkoutservice.duration_mean   0.003610
1   2024-01-20T11:12:00Z  currencyservice                 trace.from_root.row_count   4.000000
2   2024-01-20T11:12:00Z  currencyservice                trace.from_root.error_rate   0.000000
3   2024-01-20T11:12:00Z  currencyservice              trace.from_root.duration_p95   0.000072
4   2024-01-20T11:12:00Z  currencyservice             trace.from_root.duration_mean   0.000056
..                   ...              ...                                       ...        ...
45  2024-01-20T11:14:00Z  checkoutservice   trace.from_checkoutservice.duration_p95   0.005605
46  2024-01-20T11:14:00Z  checkoutservice  trace.from_checkoutservice.duration_mean   0.003674
47  2024-01-20T11:14:00Z  checkoutservice       trace.to_currencyservice.error_rate   0.000000
48  2024-01-20T11:15:00Z  currencyservice   trace.from_checkoutservice.duration_p95   0.000221
49  2024-01-20T11:15:00Z  currencyservice                 trace.from_root.row_count  12.000000

[50 rows x 4 columns],             cmdb_id                                trace_name  count    earliest_timestamp      latest_timestamp  min_value   max_value  mean_value
0   checkoutservice  trace.from_checkoutservice.duration_mean     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.003600    0.017921    0.005453
1   checkoutservice   trace.from_checkoutservice.duration_p95     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.005038    0.025790    0.006841
2   checkoutservice     trace.from_checkoutservice.error_rate     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000000    0.000000    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z  31.000000  290.000000  213.560000
4   checkoutservice    trace.to_currencyservice.duration_mean     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000121    0.006600    0.001361
5   checkoutservice     trace.to_currencyservice.duration_p95     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000203    0.018750    0.001019
6   checkoutservice       trace.to_currencyservice.error_rate     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000000    0.000000    0.000000
7   checkoutservice        trace.to_currencyservice.row_count     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z  11.000000  120.000000   87.960000
8   currencyservice  trace.from_checkoutservice.duration_mean     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000121    0.006600    0.001361
9   currencyservice   trace.from_checkoutservice.duration_p95     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000203    0.018750    0.001019
10  currencyservice     trace.from_checkoutservice.error_rate     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000000    0.000000    0.000000
11  currencyservice      trace.from_checkoutservice.row_count     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z  11.000000  120.000000   87.960000
12  currencyservice             trace.from_root.duration_mean     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000043    0.000086    0.000054
13  currencyservice              trace.from_root.duration_p95     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000051    0.000215    0.000085
14  currencyservice                trace.from_root.error_rate     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   0.000000    0.000000    0.000000
15  currencyservice                 trace.from_root.row_count     25  2024-01-20T11:12:00Z  2024-01-20T11:36:00Z   4.000000   14.000000   11.520000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_log if available, else load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp parsed as UTC datetime (Unix seconds -> UTC)
if not pd.api.types.is_datetime64_any_dtype(df_log["timestamp"]):
    df_log["timestamp"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)

# 1) Compute global P95 for each (cmdb_id, log_name) using the full series
log_p95 = (
    df_log
    .groupby(["cmdb_id", "log_name"], as_index=False)
    .agg(
        count_total=("value", "size"),
        global_p95=("value", lambda s: float(s.quantile(0.95)) if not s.empty else float("nan"))
    )
)
log_p95["global_p95"] = log_p95["global_p95"].round(6)

# 2) Filter log rows to the incident window (UTC)
start = pd.to_datetime("2024-01-20 11:09:49", utc=True)
end = pd.to_datetime("2024-01-20 11:39:49", utc=True)
df_log_window = df_log[(df_log["timestamp"] >= start) & (df_log["timestamp"] <= end)].copy()

# 3) Join the filtered rows with the global-P95 table
dfw_log = df_log_window.merge(log_p95[["cmdb_id", "log_name", "global_p95"]], on=["cmdb_id", "log_name"], how="left")

# 4) Identify anomaly rows:
# value >= global_p95 OR (global_p95 == 0 and value > 0). Ignore NaN global_p95 (treat as non-anomalous)
p95 = dfw_log["global_p95"]
anomaly_mask_log = (~p95.isna()) & ((dfw_log["value"] >= p95) | ((p95 == 0) & (dfw_log["value"] > 0)))
anomalies_log = dfw_log[anomaly_mask_log].copy()

# 5) Aggregate anomalies by (cmdb_id, log_name)
if anomalies_log.empty:
    anomaly_agg_log = pd.DataFrame(columns=[
        "cmdb_id", "log_name", "anomaly_count",
        "earliest_anomaly_timestamp", "max_value_in_window",
        "global_p95", "severity"
    ])
else:
    agg_log = (
        anomalies_log
        .groupby(["cmdb_id", "log_name"], as_index=False)
        .agg(
            anomaly_count=("value", "size"),
            earliest_anomaly_timestamp=("timestamp", "min"),
            max_value_in_window=("value", "max"),
            global_p95=("global_p95", "first")
        )
    )
    # Compute severity per rules
    def compute_sev_log(row):
        gp = row["global_p95"]
        mv = row["max_value_in_window"]
        if gp == 0:
            return float(mv)
        else:
            return float((mv - gp) / gp)
    agg_log["severity"] = agg_log.apply(compute_sev_log, axis=1)
    # Format timestamp and round numeric outputs
    agg_log["earliest_anomaly_timestamp"] = agg_log["earliest_anomaly_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    agg_log["max_value_in_window"] = agg_log["max_value_in_window"].round(6)
    agg_log["global_p95"] = agg_log["global_p95"].round(6)
    agg_log["severity"] = agg_log["severity"].round(6)
    anomaly_agg_log = agg_log.sort_values("severity", ascending=False).head(20).reset_index(drop=True)

# 6) Count distinct anomalous pairs
distinct_anomaly_pairs_log = int(anomaly_agg_log.shape[0])

# Return compact results: aggregated anomaly table (top 20) and distinct count
anomaly_agg_log, distinct_anomaly_pairs_log

```
Out[5]:
```
Summary of log anomalies within 2024-01-20 11:09:49–11:39:49 UTC

- Total distinct anomalous (cmdb_id, log_name) pairs: 20.

- Top anomalies by severity (severity, max_value_in_window, global_p95, earliest timestamp, anomaly_count):
  1) frontend | log.error_count — severity 0.071429, max=3, global_p95=2.800000, earliest=2024-01-20T11:28:00Z, anomaly_count=2
  2) frontend | log.row_count — severity 0.016780, max=2751, global_p95=2705.600000, earliest=2024-01-20T11:13:00Z, anomaly_count=2
  3) cartservice | log.row_count — severity 0.016477, max=802, global_p95=789.000000, earliest=2024-01-20T11:13:00Z, anomaly_count=2
  4) checkoutservice | log.row_count — severity 0.013917, max=102, global_p95=100.600000, earliest=2024-01-20T11:13:00Z, anomaly_count=2
  5) recommendationservice | log.row_count — severity 0.009566, max=591, global_p95=585.400000, earliest=2024-01-20T11:14:00Z, anomaly_count=2
  6) shippingservice | log.row_count — severity 0.009213, max=482, global_p95=477.600000, earliest=2024-01-20T11:14:00Z, anomaly_count=2
  7) currencyservice | log.row_count — severity 0.007395, max=2452, global_p95=2434.000000, earliest=2024-01-20T11:14:00Z, anomaly_count=2
  8) emailservice | log.row_count — severity 0.005917, max=34, global_p95=33.800000, earliest=2024-01-20T11:13:00Z, anomaly_count=2
  9) paymentservice | log.row_count — severity 0.005917, max=68, global_p95=67.600000, earliest=2024-01-20T11:13:00Z, anomaly_count=2
 10) adservice | log.row_count — severity 0.003073, max=457, global_p95=455.600000, earliest=2024-01-20T11:14:00Z, anomaly_count=2

- Additional notes:
  - Many services show modest traffic spikes (log.row_count) during the window — these are small percentage increases above their historical P95.
  - Frontend is the only service with a non-zero error_count spike (3 vs P95 2.8), which is the highest-severity log error observed in this window.
  - Several log.error_count groups report global_p95 = 0 and max_value_in_window = 0 (non-actionable zeros) — these appear in the aggregated list but do not indicate active errors.

- Immediate suggestions:
  - Investigate frontend errors around 11:28 UTC and the frontend → downstream calls occurring at the times of increased row_count.
  - Review traffic patterns for cartservice, checkoutservice, currencyservice, and recommendationservice during 11:12–11:15 UTC for potential load-related issues.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  severity
0                frontend  log.error_count              2       2024-01-20T11:28:00Z                    3         2.8  0.071429
1                frontend    log.row_count              2       2024-01-20T11:13:00Z                 2751      2705.6  0.016780
2             cartservice    log.row_count              2       2024-01-20T11:13:00Z                  802       789.0  0.016477
3         checkoutservice    log.row_count              2       2024-01-20T11:13:00Z                  102       100.6  0.013917
4   recommendationservice    log.row_count              2       2024-01-20T11:14:00Z                  591       585.4  0.009566
5         shippingservice    log.row_count              2       2024-01-20T11:14:00Z                  482       477.6  0.009213
6         currencyservice    log.row_count              2       2024-01-20T11:14:00Z                 2452      2434.0  0.007395
7            emailservice    log.row_count              2       2024-01-20T11:13:00Z                   34        33.8  0.005917
8          paymentservice    log.row_count              2       2024-01-20T11:13:00Z                   68        67.6  0.005917
9               adservice    log.row_count              2       2024-01-20T11:14:00Z                  457       455.6  0.003073
10            cartservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000
11              adservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000
12           emailservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000
13        currencyservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000
14        checkoutservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000
15         paymentservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000
16  recommendationservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000
17                  redis  log.error_count              4       2024-01-20T11:17:00Z                    0         0.0  0.000000
18                  redis    log.row_count              4       2024-01-20T11:17:00Z                    5         5.0  0.000000
19        shippingservice  log.error_count             25       2024-01-20T11:12:00Z                    0         0.0  0.000000, 20)```
```